In [12]:
from pathlib import Path
from collections import defaultdict, Counter
import re
import unicodedata
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

while not (
    (PROJECT_ROOT / "data" / "train").exists()
    and (PROJECT_ROOT / "output").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate amazon-ml-challenge project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
OUTPUT_DIR = PROJECT_ROOT / "output"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

TRAIN_S1_IDS_PATH = OUTPUT_DIR / "train_s1_ids.txt"

with open(TRAIN_S1_IDS_PATH, "r", encoding="utf-8") as f:
    train_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

print("Training S1 entities:", len(train_s1_ids))

Training S1 entities: 1986139


In [13]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize("NFKC", value)
    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()


def get_tokens(value):
    normalized = normalize_text(value)

    if not normalized:
        return []

    return normalized.split()


print(get_tokens("ABC Pvt. Ltd., Mumbai"))

['abc', 'pvt', 'ltd', 'mumbai']


In [14]:
NAME_TOKEN_THRESHOLD = 10
ADDRESS_TOKEN_THRESHOLD = 10

name_frequency = Counter()
address_frequency = Counter()
name_exact_map_train = defaultdict(list)

train_s1_entity_ids = []

row_index = 0

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):

        if row.entity_id not in train_s1_ids:
            continue

        train_s1_entity_ids.append(row.entity_id)

        name = normalize_text(row.business_name)
        address = normalize_text(row.business_address)

        name_exact_map_train[name].append(row_index)

        for token in set(get_tokens(name)):
            name_frequency[token] += 1

        for token in set(get_tokens(address)):
            address_frequency[token] += 1

        row_index += 1

print("Training S1 rows:", row_index)
print("Unique normalized names:", len(name_exact_map_train))
print("Name tokens:", len(name_frequency))
print("Address tokens:", len(address_frequency))

Training S1 rows: 1986139
Unique normalized names: 1386132
Name tokens: 106320
Address tokens: 419356


In [11]:
name_exact_map = blocking["name_exact_map"]

name_token_to_s1 = blocking["name_token_to_s1"]
address_token_to_s1 = blocking["address_token_to_s1"]

name_pair_to_s1 = blocking["name_pair_to_s1"]
address_pair_to_s1 = blocking["address_pair_to_s1"]

active_name_pairs = blocking["active_name_pairs"]
active_address_pairs = blocking["active_address_pairs"]

target_name_pair_frequency_2 = blocking[
    "target_name_pair_frequency_2"
]

target_address_pair_frequency_2 = blocking[
    "target_address_pair_frequency_2"
]

NAME_TOKEN_THRESHOLD = blocking[
    "NAME_TOKEN_FREQUENCY_THRESHOLD"
]

ADDRESS_TOKEN_THRESHOLD = blocking[
    "ADDRESS_TOKEN_FREQUENCY_THRESHOLD"
]

PAIR_S1_THRESHOLD = blocking[
    "PAIR_S1_THRESHOLD"
]

TARGET_PAIR_THRESHOLD = blocking[
    "TARGET_PAIR_THRESHOLD"
]

print(
    "Thresholds:",
    NAME_TOKEN_THRESHOLD,
    ADDRESS_TOKEN_THRESHOLD,
    PAIR_S1_THRESHOLD,
    TARGET_PAIR_THRESHOLD
)

Thresholds: 10 10 2 500


In [15]:
name_token_to_s1_train = defaultdict(list)
address_token_to_s1_train = defaultdict(list)

row_index = 0

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):

        if row.entity_id not in train_s1_ids:
            continue

        name = normalize_text(row.business_name)
        address = normalize_text(row.business_address)

        for token in set(get_tokens(name)):
            if name_frequency[token] <= NAME_TOKEN_THRESHOLD:
                name_token_to_s1_train[token].append(row_index)

        for token in set(get_tokens(address)):
            if address_frequency[token] <= ADDRESS_TOKEN_THRESHOLD:
                address_token_to_s1_train[token].append(row_index)

        row_index += 1

print(
    "Train name postings:",
    len(name_token_to_s1_train)
)

print(
    "Train address postings:",
    len(address_token_to_s1_train)
)

Train name postings: 78295
Train address postings: 367837


In [16]:
GT = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

# Keep only the 90% training S1 entities
GT_train = GT[
    GT["source1_entity_id"].isin(train_s1_ids)
].copy()

print("Training GT rows:", len(GT_train))

print(
    "Training S1 entities with GT:",
    GT_train["source1_entity_id"].nunique()
)

total_training_true_pairs = 0

for value in GT_train["matched_entity_ids"]:
    if value:
        total_training_true_pairs += len(
            [x for x in value.split(",") if x.strip()]
        )

print(
    "Total training true relationships:",
    total_training_true_pairs
)

Training GT rows: 1986139
Training S1 entities with GT: 1986139
Total training true relationships: 6874320


In [17]:
true_targets_by_s1 = {}

for row in GT_train.itertuples(index=False):

    if row.matched_entity_ids:
        matches = {
            x.strip()
            for x in row.matched_entity_ids.split(",")
            if x.strip()
        }
    else:
        matches = set()

    true_targets_by_s1[row.source1_entity_id] = matches

print(
    "S1 entities in training GT:",
    len(true_targets_by_s1)
)

match_count_distribution = Counter(
    len(v)
    for v in true_targets_by_s1.values()
)

print("\nTraining match-count distribution:")
for count in sorted(match_count_distribution):
    print(
        f"{count:>2}: "
        f"{match_count_distribution[count]:,}"
    )

S1 entities in training GT: 1986139

Training match-count distribution:
 0: 110,794
 1: 107,222
 2: 337,791
 3: 477,900
 4: 435,591
 5: 290,013
 6: 148,291
 7: 57,479
 8: 16,777
 9: 3,769
10: 481
11: 31


In [18]:
train_s1_id_to_idx = {
    entity_id: idx
    for idx, entity_id in enumerate(train_s1_entity_ids)
}

train_s1_idx_to_id = {
    idx: entity_id
    for entity_id, idx in train_s1_id_to_idx.items()
}

print("S1 index mapping:", len(train_s1_id_to_idx))

print(
    "First 5:",
    list(train_s1_idx_to_id.items())[:5]
)

S1 index mapping: 1986139
First 5: [(0, 'S1-925783039'), (1, 'S1-773889195'), (2, 'S1-377745466'), (3, 'S1-133037285'), (4, 'S1-755362802')]


In [19]:
positive_pairs = []

for s1_id, target_ids in true_targets_by_s1.items():

    s1_idx = train_s1_id_to_idx[s1_id]

    for target_id in target_ids:

        positive_pairs.append(
            (
                s1_idx,
                target_id
            )
        )

print(
    "Total positive pairs:",
    len(positive_pairs)
)

print(
    "Unique positive S1 entities:",
    len({
        s1_idx
        for s1_idx, _ in positive_pairs
    })
)

print(
    "First 5:",
    positive_pairs[:5]
)

Total positive pairs: 6874320
Unique positive S1 entities: 1875345
First 5: [(1157605, 'S2-681193310'), (1157605, 'S2-743505751'), (1157605, 'S3-775321672'), (1157605, 'S3-11291185'), (1157605, 'S3-860443364')]


In [20]:
# Select up to 2 positive relationships per training S1.
# Prefer source diversity: one S2 + one S3 when both exist.

selected_positive_pairs = []

positive_source_stats = Counter()

for s1_id, target_ids in true_targets_by_s1.items():

    if not target_ids:
        continue

    s2_targets = sorted(
        tid for tid in target_ids
        if tid.startswith("S2-")
    )

    s3_targets = sorted(
        tid for tid in target_ids
        if tid.startswith("S3-")
    )

    selected = []

    # Prefer one S2 and one S3
    if s2_targets:
        selected.append(s2_targets[0])

    if s3_targets and len(selected) < 2:
        selected.append(s3_targets[0])

    # If only one source exists, take a second deterministic positive
    if len(selected) < 2:
        remaining = sorted(
            set(target_ids) - set(selected)
        )

        if remaining:
            selected.append(remaining[0])

    s1_idx = train_s1_id_to_idx[s1_id]

    for target_id in selected:

        selected_positive_pairs.append(
            (s1_idx, target_id)
        )

        positive_source_stats[
            target_id[:2]
        ] += 1

print(
    "Selected positive pairs:",
    len(selected_positive_pairs)
)

print(
    "Unique S1 entities represented:",
    len({
        s1_idx
        for s1_idx, _ in selected_positive_pairs
    })
)

print("\nSelected positives by source:")
for source, count in sorted(positive_source_stats.items()):
    print(source, count)

Selected positive pairs: 3643468
Unique S1 entities represented: 1875345

Selected positives by source:
S2 1803108
S3 1840360


In [21]:
selected_positive_set = set(selected_positive_pairs)

# There must be no duplicates
assert len(selected_positive_pairs) == len(selected_positive_set)

# Every selected pair must be a genuine ground-truth relationship
for s1_idx, target_id in selected_positive_pairs[:10000]:

    s1_id = train_s1_idx_to_id[s1_idx]

    assert target_id in true_targets_by_s1[s1_id]

print("Positive-pair sanity checks: PASS")

print(
    "Full GT positives:",
    len(positive_pairs)
)

print(
    "Selected positives:",
    len(selected_positive_pairs)
)

print(
    "Reduction:",
    f"{len(selected_positive_pairs) / len(positive_pairs):.2%}"
)

Positive-pair sanity checks: PASS
Full GT positives: 6874320
Selected positives: 3643468
Reduction: 53.00%


In [23]:
# Fast S1-indexed ground-truth lookup.
# Reuses the existing sets instead of creating new sets.

true_targets_by_s1_idx = np.empty(
    len(train_s1_entity_ids),
    dtype=object
)

empty_targets = frozenset()
true_targets_by_s1_idx.fill(empty_targets)

for s1_id, target_ids in true_targets_by_s1.items():
    s1_idx = train_s1_id_to_idx[s1_id]
    true_targets_by_s1_idx[s1_idx] = target_ids

print(
    "S1 indexed ground-truth lookup:",
    len(true_targets_by_s1_idx)
)

print(
    "Example 0:",
    true_targets_by_s1_idx[0]
)

S1 indexed ground-truth lookup: 1986139
Example 0: {'S2-157377754', 'S3-997698194', 'S3-698172821', 'S2-517291332'}


In [24]:
from pathlib import Path

CHUNK_SIZE = 100_000

# Maximum negatives temporarily retained per source for each S1.
# We later choose at most 2 total negatives per S1.
NEGATIVES_PER_SOURCE = 2

# Number of postings sampled from a blocking list.
EXACT_POSTINGS_TO_CHECK = 3
TOKEN_POSTINGS_TO_CHECK = 1

N_S1 = len(train_s1_entity_ids)

# ------------------------------------------------------------
# Temporary negative storage
# ------------------------------------------------------------

# S2 negatives
neg_s2_num = np.full(
    (N_S1, NEGATIVES_PER_SOURCE),
    -1,
    dtype=np.int64
)

neg_s2_route = np.zeros(
    (N_S1, NEGATIVES_PER_SOURCE),
    dtype=np.int8
)

# S3 negatives
neg_s3_num = np.full(
    (N_S1, NEGATIVES_PER_SOURCE),
    -1,
    dtype=np.int64
)

neg_s3_route = np.zeros(
    (N_S1, NEGATIVES_PER_SOURCE),
    dtype=np.int8
)


def try_add_negative(
    s1_idx,
    target_num,
    route_code,
    neg_num_array,
    neg_route_array
):
    """
    Add a unique negative to the two-slot source-specific buffer.
    route_code:
        1 = exact name
        2 = name token
        3 = address token
    """

    row = neg_num_array[s1_idx]

    # Duplicate check
    if row[0] == target_num:
        if route_code < neg_route_array[s1_idx, 0]:
            neg_route_array[s1_idx, 0] = route_code
        return

    if row[1] == target_num:
        if route_code < neg_route_array[s1_idx, 1]:
            neg_route_array[s1_idx, 1] = route_code
        return

    # First free slot
    if row[0] == -1:
        row[0] = target_num
        neg_route_array[s1_idx, 0] = route_code
        return

    if row[1] == -1:
        row[1] = target_num
        neg_route_array[s1_idx, 1] = route_code


def sampled_postings(postings, target_num, limit):
    """
    Deterministically sample a small contiguous slice from
    an S1 posting list.
    """

    n = len(postings)

    if n <= limit:
        return postings

    start = target_num % n

    selected = []

    for offset in range(limit):
        selected.append(
            postings[(start + offset) % n]
        )

    return selected


def scan_hard_negatives(source_path, source_name):

    total_rows = 0
    negatives_added = 0

    if source_name == "S2":
        neg_num_array = neg_s2_num
        neg_route_array = neg_s2_route
    else:
        neg_num_array = neg_s3_num
        neg_route_array = neg_s3_route

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            target_id = row.entity_id
            target_num = int(target_id[3:])

            # ------------------------------------------------
            # 1. Exact normalized name candidates
            # ------------------------------------------------

            normalized_name = normalize_text(
                row.business_name
            )

            exact_postings = name_exact_map_train.get(
                normalized_name,
                []
            )

            for s1_idx in sampled_postings(
                exact_postings,
                target_num,
                EXACT_POSTINGS_TO_CHECK
            ):

                true_targets = true_targets_by_s1_idx[s1_idx]

                if target_id in true_targets:
                    continue

                before = (
                    neg_num_array[s1_idx, 0],
                    neg_num_array[s1_idx, 1]
                )

                try_add_negative(
                    s1_idx,
                    target_num,
                    1,
                    neg_num_array,
                    neg_route_array
                )

                after = (
                    neg_num_array[s1_idx, 0],
                    neg_num_array[s1_idx, 1]
                )

                if before != after:
                    negatives_added += 1

            # ------------------------------------------------
            # 2. Rare name-token candidate
            # ------------------------------------------------

            target_name_tokens = set(
                get_tokens(row.business_name)
            )

            best_name_token = None
            best_name_postings = None
            best_name_size = None

            for token in target_name_tokens:

                postings = name_token_to_s1_train.get(
                    token,
                    []
                )

                if not postings:
                    continue

                posting_size = len(postings)

                if (
                    best_name_size is None
                    or posting_size < best_name_size
                ):
                    best_name_token = token
                    best_name_postings = postings
                    best_name_size = posting_size

            if best_name_postings is not None:

                for s1_idx in sampled_postings(
                    best_name_postings,
                    target_num,
                    TOKEN_POSTINGS_TO_CHECK
                ):

                    true_targets = true_targets_by_s1_idx[s1_idx]

                    if target_id in true_targets:
                        continue

                    before = (
                        neg_num_array[s1_idx, 0],
                        neg_num_array[s1_idx, 1]
                    )

                    try_add_negative(
                        s1_idx,
                        target_num,
                        2,
                        neg_num_array,
                        neg_route_array
                    )

                    after = (
                        neg_num_array[s1_idx, 0],
                        neg_num_array[s1_idx, 1]
                    )

                    if before != after:
                        negatives_added += 1

            # ------------------------------------------------
            # 3. Rare address-token candidate
            # ------------------------------------------------

            target_address_tokens = set(
                get_tokens(row.business_address)
            )

            best_address_token = None
            best_address_postings = None
            best_address_size = None

            for token in target_address_tokens:

                postings = address_token_to_s1_train.get(
                    token,
                    []
                )

                if not postings:
                    continue

                posting_size = len(postings)

                if (
                    best_address_size is None
                    or posting_size < best_address_size
                ):
                    best_address_token = token
                    best_address_postings = postings
                    best_address_size = posting_size

            if best_address_postings is not None:

                for s1_idx in sampled_postings(
                    best_address_postings,
                    target_num,
                    TOKEN_POSTINGS_TO_CHECK
                ):

                    true_targets = true_targets_by_s1_idx[s1_idx]

                    if target_id in true_targets:
                        continue

                    before = (
                        neg_num_array[s1_idx, 0],
                        neg_num_array[s1_idx, 1]
                    )

                    try_add_negative(
                        s1_idx,
                        target_num,
                        3,
                        neg_num_array,
                        neg_route_array
                    )

                    after = (
                        neg_num_array[s1_idx, 0],
                        neg_num_array[s1_idx, 1]
                    )

                    if before != after:
                        negatives_added += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed | "
                f"negative assignments: {negatives_added:,}"
            )

    print(
        f"{source_name} COMPLETE | "
        f"rows={total_rows:,} | "
        f"negative assignments={negatives_added:,}"
    )


# ------------------------------------------------------------
# Run S2 and S3
# ------------------------------------------------------------

scan_hard_negatives(
    S2_PATH,
    "S2"
)

scan_hard_negatives(
    S3_PATH,
    "S3"
)

S2: 1,000,000 rows processed | negative assignments: 440,626
S2: 2,000,000 rows processed | negative assignments: 804,751
S2: 3,000,000 rows processed | negative assignments: 1,097,009
S2: 4,000,000 rows processed | negative assignments: 1,328,508
S2: 5,000,000 rows processed | negative assignments: 1,514,132
S2 COMPLETE | rows=5,034,616 | negative assignments=1,519,697
S3: 1,000,000 rows processed | negative assignments: 442,452
S3: 2,000,000 rows processed | negative assignments: 812,327
S3: 3,000,000 rows processed | negative assignments: 1,109,934
S3: 4,000,000 rows processed | negative assignments: 1,347,115
S3: 5,000,000 rows processed | negative assignments: 1,538,044
S3 COMPLETE | rows=5,285,603 | negative assignments=1,585,862


In [26]:
# Cell 13 — Build final hard-negative dataset
# This is data preparation, NOT model training.

from collections import Counter

negative_pairs = []
negative_routes = []

negatives_per_s1_counter = Counter()

for s1_idx in range(N_S1):

    selected = []

    # --------------------------------------------------------
    # Prefer one S2 negative
    # --------------------------------------------------------
    for slot in range(NEGATIVES_PER_SOURCE):

        target_num = neg_s2_num[s1_idx, slot]

        if target_num == -1:
            continue

        target_id = f"S2-{target_num}"

        if target_id in {x[1] for x in selected}:
            continue

        selected.append(
            (
                s1_idx,
                target_id,
                int(neg_s2_route[s1_idx, slot])
            )
        )

        break

    # --------------------------------------------------------
    # Prefer one S3 negative
    # --------------------------------------------------------
    for slot in range(NEGATIVES_PER_SOURCE):

        target_num = neg_s3_num[s1_idx, slot]

        if target_num == -1:
            continue

        target_id = f"S3-{target_num}"

        if target_id in {x[1] for x in selected}:
            continue

        selected.append(
            (
                s1_idx,
                target_id,
                int(neg_s3_route[s1_idx, slot])
            )
        )

        break

    # --------------------------------------------------------
    # If only one negative exists, try another from S2
    # --------------------------------------------------------
    if len(selected) < 2:

        for slot in range(NEGATIVES_PER_SOURCE):

            target_num = neg_s2_num[s1_idx, slot]

            if target_num == -1:
                continue

            target_id = f"S2-{target_num}"

            if any(target_id == x[1] for x in selected):
                continue

            selected.append(
                (
                    s1_idx,
                    target_id,
                    int(neg_s2_route[s1_idx, slot])
                )
            )

            break

    # --------------------------------------------------------
    # If still only one negative, try another from S3
    # --------------------------------------------------------
    if len(selected) < 2:

        for slot in range(NEGATIVES_PER_SOURCE):

            target_num = neg_s3_num[s1_idx, slot]

            if target_num == -1:
                continue

            target_id = f"S3-{target_num}"

            if any(target_id == x[1] for x in selected):
                continue

            selected.append(
                (
                    s1_idx,
                    target_id,
                    int(neg_s3_route[s1_idx, slot])
                )
            )

            break

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------
    for s1_idx_value, target_id, route in selected:

        negative_pairs.append(
            (s1_idx_value, target_id)
        )

        negative_routes.append(route)

    negatives_per_s1_counter[len(selected)] += 1


print("Final hard negatives:", len(negative_pairs))

print(
    "S1 with 0 negatives:",
    negatives_per_s1_counter[0]
)

print(
    "S1 with 1 negative:",
    negatives_per_s1_counter[1]
)

print(
    "S1 with 2 negatives:",
    negatives_per_s1_counter[2]
)

print(
    "Negative routes:",
    dict(Counter(negative_routes))
)

Final hard negatives: 2085899
S1 with 0 negatives: 811394
S1 with 1 negative: 263591
S1 with 2 negatives: 911154
Negative routes: {3: 698010, 1: 998227, 2: 389662}


In [27]:
# Cell 14 — Save compact training-pair representation

n_pos = len(selected_positive_pairs)
n_neg = len(negative_pairs)
n_total = n_pos + n_neg

print("Positive pairs:", n_pos)
print("Negative pairs:", n_neg)
print("Total pairs:", n_total)

# Compact arrays
pair_s1_idx = np.empty(n_total, dtype=np.int32)
pair_target_num = np.empty(n_total, dtype=np.int64)
pair_source = np.empty(n_total, dtype=np.uint8)
pair_label = np.empty(n_total, dtype=np.uint8)
pair_negative_route = np.zeros(n_total, dtype=np.int8)

# ------------------------------------------------------------
# Positives
# ------------------------------------------------------------

for i, (s1_idx, target_id) in enumerate(
    selected_positive_pairs
):

    pair_s1_idx[i] = s1_idx

    if target_id.startswith("S2-"):
        pair_source[i] = 2
    elif target_id.startswith("S3-"):
        pair_source[i] = 3
    else:
        raise ValueError(
            f"Unexpected target ID: {target_id}"
        )

    pair_target_num[i] = int(target_id[3:])
    pair_label[i] = 1


# ------------------------------------------------------------
# Negatives
# ------------------------------------------------------------

offset = n_pos

for j, (s1_idx, target_id) in enumerate(
    negative_pairs
):

    i = offset + j

    pair_s1_idx[i] = s1_idx

    if target_id.startswith("S2-"):
        pair_source[i] = 2
    elif target_id.startswith("S3-"):
        pair_source[i] = 3
    else:
        raise ValueError(
            f"Unexpected target ID: {target_id}"
        )

    pair_target_num[i] = int(target_id[3:])
    pair_label[i] = 0

    pair_negative_route[i] = negative_routes[j]


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

training_pairs_path = OUTPUT_DIR / "training_pairs_v1"

training_pairs_path.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    training_pairs_path / "s1_idx.npy",
    pair_s1_idx
)

np.save(
    training_pairs_path / "target_num.npy",
    pair_target_num
)

np.save(
    training_pairs_path / "source.npy",
    pair_source
)

np.save(
    training_pairs_path / "label.npy",
    pair_label
)

np.save(
    training_pairs_path / "negative_route.npy",
    pair_negative_route
)

print("\nSaved training-pair artifacts to:")
print(training_pairs_path)

print("\nLabels:")
print("Positive:", int((pair_label == 1).sum()))
print("Negative:", int((pair_label == 0).sum()))

print("\nSources:")
print(
    "S2:",
    int((pair_source == 2).sum())
)
print(
    "S3:",
    int((pair_source == 3).sum())
)

Positive pairs: 3643468
Negative pairs: 2085899
Total pairs: 5729367

Saved training-pair artifacts to:
/Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/output/training_pairs_v1

Labels:
Positive: 3643468
Negative: 2085899

Sources:
S2: 2819082
S3: 2910285


In [28]:
# Cell 15 — Verify saved training pairs

loaded_s1 = np.load(
    training_pairs_path / "s1_idx.npy"
)

loaded_target = np.load(
    training_pairs_path / "target_num.npy"
)

loaded_source = np.load(
    training_pairs_path / "source.npy"
)

loaded_label = np.load(
    training_pairs_path / "label.npy"
)

loaded_route = np.load(
    training_pairs_path / "negative_route.npy"
)

assert len(loaded_s1) == n_total
assert len(loaded_target) == n_total
assert len(loaded_source) == n_total
assert len(loaded_label) == n_total
assert len(loaded_route) == n_total

assert set(np.unique(loaded_label)).issubset({0, 1})
assert set(np.unique(loaded_source)).issubset({2, 3})

print("Training-pair artifact verification: PASS")

print("Rows:", len(loaded_s1))
print("Positive:", int((loaded_label == 1).sum()))
print("Negative:", int((loaded_label == 0).sum()))

Training-pair artifact verification: PASS
Rows: 5729367
Positive: 3643468
Negative: 2085899


In [29]:
# Optional memory cleanup after successful verification

del selected_positive_pairs
del negative_pairs
del negative_routes

print("Large Python pair lists released.")

Large Python pair lists released.
